In [3]:
from IPython.display import display, HTML

display(HTML("""
<style>

/* =========================
   전체 레이아웃
========================= */

div.container{
    width:85% !important;
}

div.cell.code_cell.rendered{
    width:100%;
}

div.input_prompt{
    padding:0;
}

div.prompt{
    min-width:70px;
}

div#toc-wrapper{
    padding-top:120px;
}

table.dataframe{
    font-size:12px;
}

/* =========================
   코드 입력창
========================= */

div.CodeMirror{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
    line-height:1.6;
}

/* =========================
   입력 셀
========================= */

div.input{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   코드 출력
========================= */

div.output{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   Markdown 전체
========================= */

.rendered_html{
    font-family:"마루 부리OTF 중간" !important;
    font-size:18px !important;
    line-height:1.8;
}

/* 제목 */

.rendered_html h1,
.rendered_html h2,
.rendered_html h3,
.rendered_html h4,
.rendered_html h5,
.rendered_html h6{
    font-family:"마루 부리OTF 조금굵은" !important;
}

/* 본문 */

.rendered_html p{
    font-family:"마루 부리OTF 중간" !important;
}

/* 리스트 */

.rendered_html li{
    font-family:"마루 부리OTF 중간" !important;
    padding:5px;
}

/* 인용 */

.rendered_html blockquote{
    font-family:"마루 부리OTF 중간" !important;
}

/* 표 */

.rendered_html table{
    font-family:"마루 부리OTF 중간" !important;
}

/* 코드 블록 */

.rendered_html pre,
.rendered_html code{
    font-family:"Consolas" !important;
    font-size:12pt !important;
}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))


# 감성분석: 대주주·CEO 대상

**목표:** 실제 데이터로 대주주·CEO 대상 감성 데이터를 만들고, 사람 주석 → 잠금 평가까지 이어지는 흐름을 갖춘다.

### 분석 단위와 라벨
- 한 행 = 기사 제목 1개 × 대상 인물(또는 집단) 1명 × 사건 1개
- 라벨 = 대상의 행동·사건이 주주에게 유리함/불리함/보류 -> positive / negative / neutral
- 대상이 불명확하거나 근거가 부족하거나 유리·불리가 섞이면 **HOLD(보류)**. 인물의 성격·호감도는 평가하지 않는다.

### 지키는 규칙
- `gov_dev_set.csv`(기업 라벨)는 읽지 않는다. `gov_challenge_set.csv`는 열지 않는다.
- 출처가 확실한 행만 쓴다. 조건 3가지: ① 대상 이름이 DART 명부와 정확히 일치 ② 기사 게시일 확인 ③ 게시일이 2026-01-01~기준일 사이.
- 사람이 매긴 정답이 없으면 성능 수치를 만들지 않는다.
- 점검 세트(holdout)는 라벨 합의 → 잠금 → 설정 고정 → **한 번만** 평가한다.

### 실행 순서
1. 시연 기업과 기준일
2. OpenDART로 대상자 명부 검증
3. 기사 제목 수집과 대상 연결
4. 회사 단위 분할
5. 두 명 독립 주석과 합의
6. 공개 모델 기준선
7. 점검 세트 최종 평가
8. 팀 데이터 계약 형식으로 내보내기
9. 출처
10. 결과 정리와 다음 작업

## 1. 시연 기업과 기준일

삼성전자(005930)·기준일 2026-10-27을 **잠정값**으로 쓴다.

- 근거: 설계 문서 6절의 예시가 삼성전자였고, 인계 문서도 "팀 시연 종목 확정"을 할 일로 남겼다.
- 팀이 바꾸면 아래 코드의 `DEMO`만 고치면 된다.
- 평가는 여러 업종 기업으로 넓게 한다("평가는 넓게, 시연은 좁게"). 명부 36개 기업 중 기사가 남은 곳은 33곳이다.

In [1]:
from pathlib import Path
import sys, json, subprocess
import numpy as np
import pandas as pd

PROJECT_DIR = Path(r"E:\AI_융복합수업\2차_팀프로젝트\지배구조_경영진 분석")
sys.path.insert(0, str(PROJECT_DIR / "scripts"))
import person_target_pipeline as P   # 이 단계의 모든 처리 함수(한국어 주석 포함)

DATA, OUT, ANN = PROJECT_DIR / "data", PROJECT_DIR / "outputs", PROJECT_DIR / "annotation"
AS_OF = "2026-10-27"

# 시연 기업·기준일: 팀 확정 전 잠정값. 근거 문서를 함께 남긴다.
DEMO = {
    "company": "삼성전자", "stock_code": "005930", "as_of": AS_OF,
    "status": "PROVISIONAL_TEAM_CONFIRMATION_PENDING",
    "basis": ["프롬프트/5번_경영진·지배구조분석_방향성과_삼성전자예시.md 6절",
              "프롬프트/5번_작업인계_프롬프트_20261027.md 163행"],
}

# 이 노트북이 쓰는 입력 파일. gov_dev_set.csv, gov_challenge_set.csv는 목록에 없다.
ROSTER_PATH = DATA / "person_target_roster_dart_20261027.csv"
NEWS_SEED_PATH = DATA / "person_news_seed_20261027.csv"
assert "challenge" not in " ".join(map(str, [ROSTER_PATH, NEWS_SEED_PATH]))
print(json.dumps(DEMO, ensure_ascii=False, indent=1))

{
 "company": "삼성전자",
 "stock_code": "005930",
 "as_of": "2026-10-27",
 "status": "PROVISIONAL_TEAM_CONFIRMATION_PENDING",
 "basis": [
  "프롬프트/5번_경영진·지배구조분석_방향성과_삼성전자예시.md 6절",
  "프롬프트/5번_작업인계_프롬프트_20261027.md 163행"
 ]
}


## 2. OpenDART로 대상자 명부 검증

`scripts/dart_person_roster.py`가 36개 기업에 OpenDART API 3개를 호출해 명부를 만든다.

| API | 얻는 것 | 명부에서 쓰는 곳 |
| --- | --- | --- |
| 임원 현황(`exctvSttus`) | 이름, 직위, 담당업무, 임기만료일 | CEO/대표이사 행 |
| 최대주주 현황(`hyslrSttus`) | 최대주주·특수관계인, 지분율 | 최대주주·주요주주 행 |
| 공시 목록(`list`) | 7/1~기준일 공시명 | 반기 이후 역할·지분 변동 확인 |

- 근거 보고서: 2026년 반기보고서(2026-06-30 기준). 기준일 직전의 가장 최근 정기보고서다.
- 모든 명부 행에 공시 접수번호(`rcept_no`)가 있어 원문을 되찾을 수 있다.

### 명부가 보장하는 범위
- 반기보고서는 "6월 30일 시점에 그 지위였다"까지만 보여 준다. 이후 지분 변동 신고가 있으면 `VERIFIED_ROLE_STAKE_CHANGED_AFTER_REPORT`로 표시했다.
- 7/1~10/27에 역할이 바뀐 공시는 2건(에코프로 대표이사 변경, S-Oil 최대주주 변경)이다. 기준일 시점 지위가 불분명해 해당 행은 뺐다.
- 임기 시작일은 API에 없어 비워 두었다(`valid_from`). 임기만료일(`valid_to`)은 CEO 행에만 있다.

### 데이터 전처리
- 위원회 이름('계열사대표이사 후보추천위원')을 대표이사로 잡던 문제, 각주 표시('임종룡주1)'), 띄어 쓴 이름('김 기 호')
- KB금융·POSCO홀딩스의 국민연금공단: 표기 방식 때문에 주요주주로 잘못 분류된 것을 최대주주로 고쳤다(2026-10-28).

`RUN_DART=True`로 바꾸면 API를 다시 호출한다(무료, 하루 20,000건 한도). 기본값은 저장된 결과만 읽는다.

In [2]:
RUN_DART = False   # True면 OpenDART 재호출 → 명부·원본 JSON을 새로 만든다
if RUN_DART:
    subprocess.run([sys.executable, str(PROJECT_DIR / "scripts" / "dart_person_roster.py")], check=True)

roster = pd.read_csv(ROSTER_PATH, dtype=str).fillna("")
changes = pd.read_csv(DATA / "dart_role_change_filings_20260701_20261027.csv", dtype=str).fillna("")

print("명부 행:", len(roster), "| 기업:", roster["company_name"].nunique())
print(roster.groupby(["company_name", "role"]).size().unstack(fill_value=0).to_string())
print("\n검증 상태"); print(roster["verification_status"].value_counts().to_string())
print("\n반기 이후 변동 공시(종류별)"); print(changes.groupby(["company_name", "kind"]).size().to_string())
assert roster["rcept_no"].str.len().eq(14).all(), "접수번호가 없는 명부 행이 있습니다."

명부 행: 173 | 기업: 36
role          CEO/대표이사  주요주주(특수관계인)  최대주주
company_name                             
CJ                   2            2     1
GS                   2           18     1
HLB                  1            0     1
KB금융                 1            0     1
LG에너지솔루션             1            0     1
LG화학                 1            0     1
NC                   2            0     1
POSCO홀딩스             2            0     1
S-Oil                1            0     0
SK                   2            1     1
SK텔레콤                1            0     1
고려아연                 1            2     1
금호석유화학               1            3     1
남양유업                 0            0     1
넷마블                  1            0     1
농심                   1            3     1
두산에너빌리티              3            0     1
롯데지주                 3            7     1
삼성바이오로직스             1            1     1
삼성생명                 1            5     1
삼성전자                 1            4     1
삼양식품           

In [3]:
# 대표이사 명단: 이름·직위·임기만료일·접수번호
ceo = roster.loc[roster["role"] == "CEO/대표이사", ["person_id", "company_name", "canonical_name", "role_detail", "valid_to", "rcept_no"]]
print(ceo.to_string(index=False))

person_id company_name        canonical_name                                         role_detail   valid_to       rcept_no
     P001         삼성전자                   전영현                          부회장 / ㆍ대표이사 (DS부문 경영전반 총괄) 2028-03-18 20260814003699
     P007        현대자동차                   정의선                                   회장 / 대표이사, 경영전반총괄 2028-03-19 20260814003521
     P008        현대자동차                호세 무뇨스                                      사장 / 대표이사,업무총괄 2029-03-25 20260814003521
     P009        현대자동차                   최영일                          부사장 / 대표이사, 업무총괄 (생산,안전 등) 2027-03-20 20260814003521
     P013         고려아연                   박기덕                                 사장 / 대표이사 TD사업부문 총괄 2027-03-28 20260814003958
     P017       한미사이언스                   김재교                                       대표이사 부회장 / 총괄 2028-03-26 20260814004187
     P030          카카오                   정신아                            대표이사 / 대표이사 이사후보추천위원회 위원 2028-03-26 20260819000055
     P033       

### 이름 연결 원칙
- 명부 이름이 기사 제목에 **그대로** 나올 때만 연결한다.
- 성·약칭(예: 무뇨스)이나 '오너', '총수' 같은 호칭만으로는 누구인지 확정할 수 없어 쓰지 않는다.

## 3. 기사 제목 수집과 대상 연결

### 수집 방법과 한계
- 2026-10-27 웹 검색으로 기사 **제목만** 모았다. 본문은 저작권 문제로 저장하지 않았다. 검색어·URL·수집일은 `data/person_news_seed_20261027.csv`에 있다.
- 게시일은 URL에서 읽을 수 있을 때만 채웠다.
- 계획했던 네이버 검색 API는 키가 없어 쓰지 못했다.
- 2차 수집은 점검 세트로 배정되는 기업 위주로 했다. 이때 라벨·모델 예측은 보지 않았다.
- 같은 사건을 여러 매체가 다룬 제목이 있지만, 회사가 같으면 같은 세트에 들어가므로 세트 간에 섞이지 않는다.

### 출처 기준으로 지운 기사 (2026-10-28, 90건)
| 이유 | 기사 수 |
| --- | --- |
| 게시일 미확인 | 52 |
| 제목에 대주주·CEO가 아닌 인물이 나옴 | 13 |
| 2026년 이전 기사(당시 역할 확인 불가) | 10 |
| 계열사·자회사 사건일 가능성 | 9 |
| 명부 이름과 정확히 일치하지 않음 | 6 |

- 이유가 여럿이면 표의 위쪽 이유 하나로 셌다.

### 연결 규칙 (`P.link_targets`)
명부 이름이 제목에 그대로 있고, 게시일이 2026-01-01~기준일 사이일 때만 행을 만든다(`LINKED_DART`). 나머지는 행을 만들지 않는다.

In [4]:
news = pd.read_csv(NEWS_SEED_PATH, dtype=str).fillna("")
targets = P.link_targets(news, roster, as_of=AS_OF)

print("기사:", len(news), "| 대상 행:", len(targets))
print(targets["link_status"].value_counts().to_string())
print("\n날짜 확인 상태"); print(targets["date_status"].value_counts().to_string())
assert targets["link_status"].isin(P.IN_SCOPE).all(), "범위 밖 대상 행이 있습니다."

기사: 30 | 대상 행: 31
LINKED_DART    31

날짜 확인 상태
DATE_FROM_URL    31


## 4. 회사 단위 분할 (주석 전에 고정)

데이터를 **dev(설정 고르기용)**와 **holdout(마지막에 한 번만 쓰는 점검 세트)**으로 나눈다.

**왜 회사 단위인가?** 같은 회사가 양쪽에 있으면 모델이 그 회사·인물을 "외운" 효과까지 성능으로 잡힌다. 처음 보는 회사에서도 통하는지 보려면 회사 전체를 한쪽에만 둔다.

**왜 해시인가?** 회사 이름을 SHA-256 해시로 바꾼 값으로 약 35%를 holdout에 보낸다. 해시는 같은 입력이면 늘 같은 값이 나오므로, 회사를 더 모아도 기존 배정이 바뀌지 않는다. 해시에 섞는 고정 문자열(소금값, salt)과 비율은 라벨을 보기 전에 정했다.

In [5]:
split = P.assign_company_split(targets["company_name"])
targets["split"] = targets["company_name"].map(split)
print(json.dumps(split, ensure_ascii=False, indent=1))
print(targets.groupby(["split", "company_name"]).size().to_string())

# 대상 행 전체 저장(출처·연결 근거 포함). 기존 파일이 있으면 덮어쓰지 않는다.
CAND_PATH = DATA / "person_target_candidates_20261027.csv"
if not CAND_PATH.exists():
    targets.to_csv(CAND_PATH, index=False, encoding="utf-8-sig")
    print("\n저장:", CAND_PATH.name)
else:
    print("\n기존 후보 파일 보존:", CAND_PATH.name)

# 분할 기록: holdout 문장 내용의 해시를 남겨 이후 바뀌지 않았는지 확인할 수 있게 한다
SPLIT_MANIFEST = DATA / "person_target_split_manifest_20261027.json"
hold = targets.loc[targets["split"] == "holdout"]
manifest = {"method": "sha256(salt|company) % 100 < pct", "salt": P.HOLDOUT_SALT, "pct": P.HOLDOUT_PCT,
            "assignment": split, "holdout_rows": int(len(hold)),
            "holdout_unlabeled_sha256": P.sha256_of(hold, ["row_id", "text_original", "target_name"]),
            "labels_seen_before_split": False, "revision": "r2-scope-cleanup"}
if not SPLIT_MANIFEST.exists():
    SPLIT_MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("holdout 주석 대상 행:", len(hold), "| dev:", int((targets["split"] == "dev").sum()))

{
 "CJ": "holdout",
 "GS": "holdout",
 "LG화학": "dev",
 "SK": "holdout",
 "SK텔레콤": "holdout",
 "롯데지주": "holdout",
 "삼성생명": "holdout",
 "삼성전자": "dev",
 "삼양식품": "holdout",
 "셀트리온": "dev",
 "에코프로": "holdout",
 "영풍": "holdout",
 "우리금융지주": "holdout",
 "카카오": "dev",
 "태광산업": "holdout",
 "하이브": "holdout",
 "한국앤컴퍼니": "holdout",
 "한화": "holdout"
}
split    company_name
dev      LG화학            1
         삼성전자            2
         셀트리온            1
         카카오             1
holdout  CJ              1
         GS              2
         SK              4
         SK텔레콤           1
         롯데지주            1
         삼성생명            3
         삼양식품            1
         에코프로            1
         영풍              1
         우리금융지주          1
         태광산업            2
         하이브             5
         한국앤컴퍼니          2
         한화              1

저장: person_target_candidates_20261027.csv
holdout 주석 대상 행: 26 | dev: 5


### 결과 해석
- 현재 holdout은 14개사 26행, dev는 4개사 5행이다.
- **holdout 목표 100행에 크게 못 미친다.** 이 규모로는 클래스별 성능을 믿을 수 없다. 게시일을 확인할 수 있는 출처로 더 모아야 한다.
- dev 5행으로는 아래 기준선의 설정을 고를 수 없다. dev 수집이 먼저다.

## 5. 두 명 독립 주석과 합의

규약: `annotation/` 폴더의 주석 규약 v1.0 문서

- 주석자 A·B가 각자 시트를 채운다. 행 순서는 사람마다 다르게 섞었다.
- 시트에는 모델 예측·기업 라벨이 **없다**. 주석자는 `outputs/`의 기준선 파일을 열지 않는다.
- 이미 채운 시트는 절대 덮어쓰지 않는다.
- 둘 다 끝나면 `P.compare_annotators()`가 일치도(kappa)와 불일치 목록을 만든다. 불일치는 규약 조항을 근거로 합의하고 이유를 적는다.

**Cohen's kappa(코헨 카파):** 두 사람이 우연히 같은 답을 고를 확률을 빼고 계산한 일치도다. 1이면 완전 일치, 0이면 우연 수준이다. 0.6 이상이면 다음 단계로 간다.

In [6]:
for name, status in P.write_annotation_sheets(targets, ANN):
    print(f"- {name}: {status}")

def sheet_done(path):
    # 주석이 하나라도 채워졌는지 본다. 빈 시트면 False
    df = pd.read_csv(path, dtype=str).fillna("")
    return bool((df["person_impact_label"].ne("") | df["review_status"].ne("")).any())

agreement = {}
for split_name in ["dev", "holdout"]:
    a, b = ANN / f"annotator_A_{split_name}.csv", ANN / f"annotator_B_{split_name}.csv"
    if sheet_done(a) and sheet_done(b):
        merged, report, disagree = P.compare_annotators(a, b)
        agreement[split_name] = report
        adj_path = ANN / f"adjudication_{split_name}.csv"
        if not adj_path.exists():
            disagree.to_csv(adj_path, index=False, encoding="utf-8-sig")
        print(split_name, json.dumps(report, ensure_ascii=False))
    else:
        agreement[split_name] = "NOT_RUN: 주석 시트가 비어 있음"
        print(f"{split_name}: 주석 전 → kappa·합의 미실행")

- annotator_A_dev.csv: 5행 생성
- annotator_B_dev.csv: 5행 생성
- annotator_A_holdout.csv: 26행 생성
- annotator_B_holdout.csv: 26행 생성
dev: 주석 전 → kappa·합의 미실행
holdout: 주석 전 → kappa·합의 미실행


## 6. 공개 모델 기준선 (dev)

**기준선(baseline)**은 "따로 학습하지 않은 공개 모델이 이 정도 한다"는 비교 기준이다.

- 모델: `snunlp/KR-FinBert-SC`(한국어 금융 문장을 긍정·중립·부정으로 분류). 추가 학습은 하지 않는다.
- 입력 두 가지: 제목만(`text`) / 대상 표식을 붙인 제목(`marked`)
- 판정: 가장 높은 확률이 기준값 `tau`보다 낮으면 HOLD
- 입력 방식과 `tau`는 **GroupKFold**로 고른다. 회사를 묶음 단위로 나눠 돌아가며 검증하는 방법이라, 같은 회사가 학습·검증에 함께 들어가지 않는다.

**주의:** 이 모델은 문장 전체의 분위기를 판단한다. 대상 인물별로 판단하는 이 작업과 정의가 달라 점수가 낮게 나와도 이상하지 않다. 모델 카드의 정확도를 이 작업 성능으로 쓰지 않는다.

아래 셀은 dev 확률만 미리 계산해 저장한다. **정답이 없으면 지표는 계산하지 않는다.**

In [7]:
dev = targets.loc[targets["split"] == "dev"].reset_index(drop=True)
PROBA_PATH = OUT / "person_target_dev_baseline_proba_20261027.csv"

proba = {v: P.predict_proba([P.build_input(r, v) for _, r in dev.iterrows()]) for v in ["text", "marked"]}
proba_df = dev[["row_id", "company_name", "target_name", "text_original"]].copy()
for v, arr in proba.items():
    for i, lab in enumerate(P.LABELS):
        proba_df[f"{v}_p_{lab[:3]}"] = arr[:, i].round(4)
proba_df.to_csv(PROBA_PATH, index=False, encoding="utf-8-sig")
print("dev 확률 저장:", PROBA_PATH.name, "| 행:", len(proba_df))
print("(주석자에게 보여 주지 않는 파일입니다)")

dev 확률 저장: person_target_dev_baseline_proba_20261027.csv | 행: 5
(주석자에게 보여 주지 않는 파일입니다)


In [8]:
# 지표 함수가 의도대로 동작하는지 가짜 배열로 확인한다. 아래 숫자는 성능이 아니다.
_g = ["positive", "negative", "neutral", "HOLD"]
_c = ["X", "X", "Y", "Y"]
_r = P.evaluate(_g, ["positive", "negative", "neutral", "HOLD"], _c)
assert _r["macro_f1_strict"] == 1.0 and _r["n_gold_hold"] == 1 and _r["hold_rate"] == 0.0
_r = P.evaluate(_g, ["HOLD", "negative", "neutral", "HOLD"], _c)
assert _r["hold_rate"] == 1/3 and _r["per_class"]["positive"]["recall"] == 0.0   # 보류는 오답으로 센다
assert P.decide(pd.DataFrame([[0.2, 0.3, 0.5]]).values, 0.6)[0] == "HOLD"
print("지표 함수 자체 점검 통과 (가짜 배열, 성능 수치 아님)")

지표 함수 자체 점검 통과 (가짜 배열, 성능 수치 아님)


In [9]:
# dev 정답이 합의로 확정됐을 때만 기준선 지표를 계산한다.
DEV_FINAL = ANN / "final_labels_dev.csv"   # finalize_labels() 결과를 저장할 경로
baseline_report = "NOT_RUN: 사람 dev 정답 없음"
if DEV_FINAL.exists():
    gold = dev[["row_id"]].merge(pd.read_csv(DEV_FINAL, dtype=str), on="row_id", how="left")["final_decision"]
    cv, oof = P.group_cv_select(gold.values, proba, dev["company_name"].values)
    baseline_report = cv
    dev_eval = dev.assign(gold=gold.values, pred=oof)
    err, err_summary = P.error_table(dev_eval, "gold", "pred")
    err.to_csv(OUT / "person_target_dev_errors.csv", index=False, encoding="utf-8-sig")
    print(json.dumps(cv, ensure_ascii=False, indent=1)); print(err_summary.to_string(index=False))
else:
    print(baseline_report)

NOT_RUN: 사람 dev 정답 없음


## 7. 점검 세트 최종 평가

### 실행 조건 (모두 만족해야 함)
1. holdout 주석 두 명 완료, kappa 보고, 불일치 합의 완료
2. `P.lock_holdout()`으로 잠금 파일과 해시 기록
3. dev에서 고른 설정을 `P.freeze_config()`로 고정
4. 결과 파일이 아직 없음(있으면 재실행 거부)
5. 사람이 직접 `RUN_FINAL_EVAL = True`로 바꿈

### 보고 항목
- **macro-F1(strict):** 클래스별 F1을 단순 평균한 값. HOLD로 보류한 행은 오답으로 센다.
- 클래스별 정밀도·재현율·F1, 회사별 macro-F1
- 보류율, 판정한 행의 정확도, 정답이 HOLD인 행 수, 혼동행렬

In [10]:
HOLDOUT_LOCKED = DATA / "person_target_locked_holdout.csv"
LOCK_MANIFEST = DATA / "person_target_locked_holdout.manifest.json"
FROZEN_CONFIG = OUT / "person_target_frozen_config.json"
FINAL_RESULT = OUT / "person_target_final_holdout_eval.json"
RUN_FINAL_EVAL = False   # 조건을 모두 확인한 사람이 직접 True로 바꾼다

gate = {"holdout_locked": HOLDOUT_LOCKED.exists() and LOCK_MANIFEST.exists(),
        "config_frozen": FROZEN_CONFIG.exists(),
        "not_yet_evaluated": not FINAL_RESULT.exists(),
        "run_flag": RUN_FINAL_EVAL}
print(json.dumps(gate, ensure_ascii=False))
if all(gate.values()):
    final = P.final_eval_once(HOLDOUT_LOCKED, LOCK_MANIFEST, FROZEN_CONFIG, FINAL_RESULT)
    print(json.dumps(final, ensure_ascii=False, indent=1))
else:
    print("최종 평가 미실행: 조건 미충족")

{"holdout_locked": false, "config_frozen": false, "not_yet_evaluated": true, "run_flag": false}
최종 평가 미실행: 조건 미충족


## 8. 팀 데이터 계약 형식으로 내보내기

### 왜 필요한가
이 결과는 다음 단계인 Verifier(근거 확인)와 조율 단계(일치·충돌·보류 정리)로 넘어간다. 두 단계가 읽을 수 있도록 팀이 정한 형식(데이터 계약)으로 내보낸다. 형식 정의: `schema/governance_agent_output.schema.json`(초안 0.1)

### 주요 필드
| 계약 항목 | 필드 |
| --- | --- |
| 식별자 | `reportId`, `claims[].claimId`, `evidence[].evidenceId` |
| 주장 종류(사실·해석·예측) | `claimType`: fact · interpretation · forecast |
| 반대 근거 | `claims[].counterEvidence` |
| 미확인 사항 | `unresolved`, `assumptions` |
| 기준일·공개시각 | `asOf`, 근거별 `dataDate`·`publishedAt` |
| 출처 태그 | `evidence[].sourceTag` 예: `[공시 20260814003699]` |

### 내보내는 내용
- 감성은 **아직 검증 전**이다. 그래서 모든 감성 주장을 `label=HOLD`, `reportStatus=hold`, `validationStatus=UNVALIDATED`로 내보낸다. 모델 확률은 참고값이다.
- 감성 주장은 규칙 판정을 바꾸지 않는다. 규칙 판정은 `지배구조판정_삼성전자.ipynb`가 맡는다.
- 변환: `P.to_contract_sentiment()` / 검사: JSON Schema 형식 검사 → `schema/validate_output.py` 규칙 검사(미래정보, 근거 없는 숫자, 출처 태그 일치 등)

In [11]:
from datetime import datetime, timezone, timedelta
from jsonschema import Draft202012Validator

SCHEMA_DIR = PROJECT_DIR / "schema"
sys.path.insert(0, str(SCHEMA_DIR))
import validate_output as V   # 계약 규칙 검사기(형식 검사 뒤에 실행)

SCHEMA = json.loads((SCHEMA_DIR / "governance_agent_output.schema.json").read_text(encoding="utf-8"))

def check(doc):
    # 형식 오류가 있으면 그것부터, 없으면 계약 규칙 오류를 돌려준다. 빈 목록이면 통과
    errs = [e.message for e in Draft202012Validator(SCHEMA).iter_errors(doc)]
    return errs or V.contract_errors(doc)

# 규칙 판정 쪽 예시(삼성전자 주장 7개)가 계약을 지키는지 먼저 확인한다
example = json.loads((SCHEMA_DIR / "example_삼성전자_20261027.json").read_text(encoding="utf-8"))
print("규칙 판정 예시:", "통과" if not check(example) else check(example))

규칙 판정 예시: 통과


In [12]:
# 시연 기업의 dev 대상 행을 계약 형식의 감성 주장(S)으로 바꾼다
pos = np.where(dev["company_name"].values == DEMO["company"])[0]
demo_rows = dev.iloc[pos]
s_claims, s_evidence, s_unresolved = P.to_contract_sentiment(
    demo_rows, proba["marked"][pos], roster,
    method="KR-FinBert-SC, 대상 표식 입력(marked), HOLD 임계값 미정")

corp = json.loads((DATA / "dart_raw" / "corpCode_selected.json").read_text(encoding="utf-8"))[DEMO["stock_code"]]
contract = {
    "schemaVersion": "0.1-draft",
    "reportId": f"RPT-EXAMPLE-{AS_OF.replace('-', '')}-{DEMO['stock_code']}-S",   # 실제 값은 Orchestrator 발급(팀 확인)
    "agentId": "agent5_governance",                                                # 번호는 팀 확인
    "perspective": "경영진·대주주",
    "subject": {"companyName": DEMO["company"], "stockCode": DEMO["stock_code"], "corpCode": corp["corp_code"]},
    "asOf": AS_OF,
    "generatedAt": datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="seconds"),
    "dataMode": "DART_PARTIAL",
    "completeness": "partial",
    "claims": s_claims, "evidence": s_evidence, "unresolved": s_unresolved,
    "assumptions": [], "handoff": [],
    "qualityFlags": ["SENTIMENT_UNVALIDATED", "PERSON_SENTIMENT_ONLY"],
    "disclaimer": "투자 권유가 아니다. 감성 신호는 검증 전이며 규칙 판정을 바꾸지 않는다.",
}
errs = check(contract)
assert not errs, errs
CONTRACT_PATH = OUT / f"contract_sentiment_{DEMO['company']}_{AS_OF.replace('-', '')}.json"
CONTRACT_PATH.write_text(json.dumps(contract, ensure_ascii=False, indent=2), encoding="utf-8")

print("계약 검사 통과 | 감성 주장", len(s_claims), "| 근거", len(s_evidence), "| 미확인", len(s_unresolved))
print(pd.DataFrame([{"claimId": c["claimId"], "대상": c["sentiment"]["target"]["name"],
                     "역할": c["sentiment"]["target"]["role"], "근거": ",".join(c["evidenceIds"]),
                     **{f"p_{k[:3]}": v for k, v in c["sentiment"]["probabilities"].items()}}
                    for c in s_claims]).to_string(index=False))
print("저장:", CONTRACT_PATH.name)

계약 검사 통과 | 감성 주장 2 | 근거 3 | 미확인 1
claimId  대상          역할        근거  p_neg  p_neu  p_pos
    S01 이재용 주요주주(특수관계인) N003,R004 0.0249 0.6460 0.3291
    S02 이재용 주요주주(특수관계인) N004,R004 0.0038 0.9953 0.0009
저장: contract_sentiment_삼성전자_20261027.json


In [13]:
# 검사기가 계약 위반을 실제로 잡는지 시험한다(원본은 그대로 두고 복사본만 바꾼다)
import copy

def mutated(doc, change):
    d = copy.deepcopy(doc)
    change(d)
    return d

ev_by_id = lambda d, eid: next(e for e in d["evidence"] if e["evidenceId"] == eid)
cases = {
    "미래정보(공개시각 > 기준일)": mutated(example, lambda d: ev_by_id(d, "E03").update(publishedAt="2026-11-01")),
    "숫자 생성(근거에 없는 숫자)": mutated(example, lambda d: next(c for c in d["claims"] if c["claimId"] == "P01").update(statement="증여 후 지분율은 1.9%다.")),
    "사실 주장에 신호": mutated(example, lambda d: d["claims"][0].update(signal="긍정")),
    "출처 태그 불일치": mutated(example, lambda d: ev_by_id(d, "E03").update(sourceTag="[공시 20260814003699]")),
    "검증 전 감성을 확인 사실로": mutated(contract, lambda d: d["claims"][0].update(reportStatus="confirmed_fact")),
    "없는 근거 ID": mutated(example, lambda d: d["claims"][2].update(evidenceIds=["E99"])),
    "보류 사유 누락": mutated(contract, lambda d: d["claims"][0].pop("holdReason")),
    "게시일 없는 근거를 미확인에 안 올림": mutated(contract, lambda d: d["evidence"][0].update(publishedAt=None, publishedAtSource="UNKNOWN")),
}
result = pd.DataFrame([{"시험": k, "잡은 오류 수": len(check(v)), "첫 오류": (check(v) or ["(못 잡음)"])[0][:60]}
                       for k, v in cases.items()])
print(result.to_string(index=False))
assert (result["잡은 오류 수"] > 0).all(), "검사기가 놓친 위반이 있습니다."

                  시험  잡은 오류 수                                                         첫 오류
    미래정보(공개시각 > 기준일)        1              E03: 공개시각 2026-11-01이 기준일 2026-10-27보다 늦음(미래정보)
    숫자 생성(근거에 없는 숫자)        1                                     P01: 문장 숫자 1.9이 근거 값에 없음
           사실 주장에 신호        1 {'claimId': 'F01', 'claimType': 'fact', 'kind': 'disclosed_f
           출처 태그 불일치        1                        E03: 출처 태그 [공시 20260814003699] ≠ [보도]
     검증 전 감성을 확인 사실로        1                                  S01: 검증 전 감성은 확인 사실이 될 수 없음
            없는 근거 ID        1                                               G08: 없는 근거 E99
            보류 사유 누락        1                          'holdReason' is a required property
게시일 없는 근거를 미확인에 안 올림        1             N003: 공개시각을 모르면 unresolved에 DATE_UNKNOWN으로 올려야 함


## 9. 출처

이 분석에 쓴 자료와 문서다. 수치·라벨·판단마다 어디서 왔는지 이 표로 되짚을 수 있다.

### 프로젝트 내부 문서
| 문서 | 쓴 곳 |
| --- | --- |
| `프롬프트/5번_경영진·지배구조분석_방향성과_삼성전자예시.md` | 시연 기업·기준일(잠정), 13개 판정 규칙, 감성 경계 초안 |
| `프롬프트/5번_작업인계_프롬프트_20261027.md` | 시연 종목 미확정 확인, 기준일 필터 원칙 |
| `프롬프트/5번_예선기획안_초안_20261027.md` | "평가는 넓게, 시연은 좁게", 네이버 검색 API 수집 계획 |
| `보고서 및 txt/KIC_Agent_1_3_기초기획서_R4.docx` | 팀 데이터 계약 항목, 일정(10/28 스키마 확정 ~ 11/12 제출), 감성 모델 리스크 |
| 팀 구조도(Planner·Researcher·Verifier·Orchestrator, 7개 관점), 2026-10-28 공유 | 이 관점의 위치와 출력이 넘어가는 곳 |
| `지배구조판정_삼성전자.ipynb`, `agent_memo_governance_삼성전자.json` | AgentMemo 원본, 계약 예시의 근거값(E03·E14·E15) |

### 공시·데이터
| 자료 | 주소 | 쓴 곳 |
| --- | --- | --- |
| OpenDART 개발가이드 - 공시정보(공시검색, 고유번호) | https://opendart.fss.or.kr/guide/main.do?apiGrpCd=DS001 | `list.json`, `corpCode.xml` |
| OpenDART 개발가이드 - 정기보고서 주요정보(임원 현황, 최대주주 현황) | https://opendart.fss.or.kr/guide/main.do?apiGrpCd=DS002 | `exctvSttus.json`, `hyslrSttus.json` |
| DART 공시 원문 | `https://dart.fss.or.kr/dsaf001/main.do?rcpNo=<접수번호>` | 명부 행별 근거(아래 코드 셀 목록) |
| 기사 제목 | 웹 검색 결과의 제목·URL, 수집일 2026-10-27 | 대상 행 원문(아래 코드 셀 목록) |

### 모델·도구
| 자료 | 주소 | 쓴 곳 |
| --- | --- | --- |
| KR-FinBert-SC 모델 카드(커밋 f8586286cc3161fb648e9fee09a456069fd846d0) | https://huggingface.co/snunlp/KR-FinBert-SC | 기준선 확률 |
| scikit-learn `cohen_kappa_score` | https://scikit-learn.org/stable/modules/generated/sklearn.metrics.cohen_kappa_score.html | 주석자 일치도 |
| scikit-learn `GroupKFold` | https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html | 회사 단위 교차검증 |
| scikit-learn `f1_score` | https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html | macro-F1 |
| JSON Schema Draft 2020-12 | https://json-schema.org/draft/2020-12 | 출력 스키마 문법 |
| python-jsonschema | https://python-jsonschema.readthedocs.io/ | 형식 검사 |

### 방법 근거
- Cohen, J. (1960). A coefficient of agreement for nominal scales. *Educational and Psychological Measurement*, 20(1), 37–46. — kappa 정의
- Landis, J. R., & Koch, G. G. (1977). The measurement of observer agreement for categorical data. *Biometrics*, 33(1), 159–174. — kappa 0.61 이상을 '상당한 일치'로 보는 관례. 주석 규약의 진행 기준(0.6)이 여기서 왔다.

### 출처가 없는 항목
- 사람 정답 라벨: 아직 없다. 주석 시트는 비어 있다.
- 판정 기준값(G 규칙의 97%·50%·95% 등): 팀 합의 전 초안이다.

In [14]:
# 실제로 쓴 DART 보고서와 기사 목록. 출처 표의 '아래 코드 셀 목록'이 이 출력이다
from urllib.parse import urlparse

reports = (roster.groupby(["company_name", "rcept_no"]).size().rename("명부 행").reset_index()
           .assign(공시=lambda d: "2026 반기보고서", 원문=lambda d: "https://dart.fss.or.kr/dsaf001/main.do?rcpNo=" + d["rcept_no"]))
print(f"DART 반기보고서 {len(reports)}건(회사 {reports['company_name'].nunique()}곳)")
print(reports.to_string(index=False))
print(f"\n반기 이후 변동 공시 {len(changes)}건:", changes["kind"].value_counts().to_dict())

news["매체"] = news["source_url"].map(lambda u: urlparse(u).netloc.replace("www.", "").replace("m.", "", 1) if u.startswith("http") else "")
print(f"\n기사 {len(news)}건, 매체 {news['매체'].nunique()}곳")
print(news["매체"].value_counts().to_string())
print("\n기사 전체 목록")
print(news[["news_id", "company_name", "source_title", "source_url"]].to_string(index=False))

DART 반기보고서 36건(회사 36곳)
company_name       rcept_no  명부 행         공시                                                          원문
          CJ 20260814002852     5 2026 반기보고서 https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20260814002852
          GS 20260814003779    21 2026 반기보고서 https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20260814003779
         HLB 20260814002713     2 2026 반기보고서 https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20260814002713
        KB금융 20260814004200     2 2026 반기보고서 https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20260814004200
    LG에너지솔루션 20260813000255     2 2026 반기보고서 https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20260813000255
        LG화학 20260814002655     2 2026 반기보고서 https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20260814002655
          NC 20260814003764     3 2026 반기보고서 https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20260814003764
    POSCO홀딩스 20260814000957     3 2026 반기보고서 https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20260814000957
       S-Oil 20260814000827     

## 10. 결과 정리와 다음 작업

### 다음 수집 목표 (팀 합의 필요)
- holdout을 26행에서 100행 이상으로, dev를 5행에서 100행 이상으로 늘린다.
- 게시일을 확인할 수 있는 출처로 모은다. 네이버 검색 API 키를 받으면 같은 seed CSV 형식으로 수집한다.

### 계약 형식에서 팀 확인이 필요한 것
reportId 발급 주체, claimId 형식, 에이전트 번호, 신호 어휘 통일 여부, 시각 형식, 출처 태그 문자열, 투자심리·수급 관점과의 감성 경계

In [15]:
status = {
    "notebook": "감성분석_대주주CEO대상.ipynb",
    "as_of": AS_OF, "demo": DEMO,
    "executed": {
        "dart_roster_rows": int(len(roster)), "dart_companies": int(roster["company_name"].nunique()),
        "news_titles": int(len(news)), "target_rows": int(len(targets)),
        "link_status_counts": targets["link_status"].value_counts().to_dict(),
        "split": split,
        "rows_by_split": targets["split"].value_counts().to_dict(),
        "dev_baseline_proba_rows": int(len(proba_df)),
        "contract_output": CONTRACT_PATH.name, "contract_claims": len(s_claims), "contract_check": "PASS",
        "validator_negative_tests": f"{int((result['잡은 오류 수'] > 0).sum())}/{len(result)} caught",
    },
    "not_executed": {
        "team_confirmation_of_demo": "팀 확인 필요",
        "human_annotation": "주석자 2명 미배정",
        "kappa_and_adjudication": agreement,
        "dev_baseline_metrics": baseline_report if isinstance(baseline_report, str) else "RUN",
        "final_holdout_eval": "NOT_RUN" if not FINAL_RESULT.exists() else "RUN",
    },
    "challenge_set_opened": False,
    "company_labels_used_as_person_gold": False,
}
(OUT / "person_target_stage2_status.json").write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(status, ensure_ascii=False, indent=1))

{
 "notebook": "감성분석_대주주CEO대상.ipynb",
 "as_of": "2026-10-27",
 "demo": {
  "company": "삼성전자",
  "stock_code": "005930",
  "as_of": "2026-10-27",
  "status": "PROVISIONAL_TEAM_CONFIRMATION_PENDING",
  "basis": [
   "프롬프트/5번_경영진·지배구조분석_방향성과_삼성전자예시.md 6절",
   "프롬프트/5번_작업인계_프롬프트_20261027.md 163행"
  ]
 },
 "executed": {
  "dart_roster_rows": 173,
  "dart_companies": 36,
  "news_titles": 30,
  "target_rows": 31,
  "link_status_counts": {
   "LINKED_DART": 31
  },
  "split": {
   "CJ": "holdout",
   "GS": "holdout",
   "LG화학": "dev",
   "SK": "holdout",
   "SK텔레콤": "holdout",
   "롯데지주": "holdout",
   "삼성생명": "holdout",
   "삼성전자": "dev",
   "삼양식품": "holdout",
   "셀트리온": "dev",
   "에코프로": "holdout",
   "영풍": "holdout",
   "우리금융지주": "holdout",
   "카카오": "dev",
   "태광산업": "holdout",
   "하이브": "holdout",
   "한국앤컴퍼니": "holdout",
   "한화": "holdout"
  },
  "rows_by_split": {
   "holdout": 26,
   "dev": 5
  },
  "dev_baseline_proba_rows": 5,
  "contract_output": "contract_sentiment_삼성전자_20261027.json"